# House price forcast

## 1. Import Bibliotek

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy.stats import shapiro
from statsmodels.tools import eval_measures

Source of data: https://www.kaggle.com/harlfoxem/housesalesprediction  
link -> Discussion -> Column definitions

--- 
### ZADANIA
ZAD1: Wczytaj dane, zapoznaj się z nimi (braki, wartości odstające), wyświetl wykresy zależności i policz korelacje.

ZAD2: Zbuduj model regresji liniowej, spróbuj osiągnąć wysoki wynik R^2, zbadaj reszty.

---

## KROK 1: Import Danych

In [ ]:
try:
    houses = pd.read_csv("data/kc_house_data.csv")
except FileNotFoundError:
    try:
        houses = pd.read_csv("../data/kc_house_data.csv")
    except FileNotFoundError:
        print("Nie znaleziono pliku. Upewnij się, że jesteś w odpowiednim katalogu.")
        houses = pd.DataFrame() # Pusty dataframe, żeby uniknąć błędów

### Przegląd danych

In [ ]:
if not houses.empty:
    print("Info:")
    print(houses.info())
    print("\nDescribe:")
    print(houses.describe())
    print("\nBraki danych:")
    print(houses.isnull().sum())

### Korelacja i Wykres (Cena vs Powierzchnia)

In [ ]:
if not houses.empty:
    plt.figure(figsize=(10, 6))
    sns.scatterplot(x='sqft_living', y='price', data=houses)
    plt.title('Zależność ceny od powierzchni (sqft_living)')
    plt.show()

    corr = houses['price'].corr(houses['sqft_living'])
    print(f"\nKorelacja price vs sqft_living: {corr}")

## KROK 2: Przygotowanie Danych - Data Wrangling

In [ ]:
if not houses.empty:
    houses['date'] = pd.to_datetime(houses['date'])
    houses['yr_sold'] = houses['date'].dt.year
    houses['age'] = houses['yr_sold'] - houses['yr_built']
    # Korekta ewentualnych ujemnych wartości wieku (błędy danych lub renowacje traktowane jako budowa)
    houses['age'] = houses['age'].apply(lambda x: x if x >= 0 else 0)

    # Dodatkowa zmienna - luksusowe dzielnice
    # Definiujemy luksusowe dzielnice jako te w górnym kwartylu średnich cen
    zip_price = houses.groupby('zipcode')['price'].mean()
    luxury_threshold = zip_price.quantile(0.75)
    luxury_zips = zip_price[zip_price > luxury_threshold].index
    
    houses['is_luxury'] = houses['zipcode'].apply(lambda x: 1 if x in luxury_zips else 0)

### Wizualizacja Danych (Cena vs Grade)

In [ ]:
if not houses.empty:
    plt.figure(figsize=(10,6))
    sns.boxplot(x='grade', y='price', data=houses)
    plt.title("Cena względem oceny konstrukcji (grade)")
    plt.show()

## KROK 4: Uczenie Modelu - Model Training

In [ ]:
if not houses.empty:
    # Model 1
    # Wybieramy zmienne, które wydają się istotne: sqft_living, grade, age, waterfront
    formula1 = "price ~ sqft_living + grade + age + C(waterfront)"
    model1 = smf.ols(formula=formula1, data=houses)
    result1 = model1.fit()

    # Model 2
    # Próbujemy poprawić wynik logarytmując cenę (częsty zabieg przy cenach nieruchomości)
    # oraz dodając zmienną luksusu i widoku
    houses['log_price'] = np.log(houses['price'])
    
    formula2 = "log_price ~ sqft_living + grade + age + C(waterfront) + is_luxury + view + bathrooms"
    model2 = smf.ols(formula=formula2, data=houses)
    result2 = model2.fit()

    # Model 3 - Ulepszony
    # Dodajemy interakcje i zmienne lokalizacyjne oraz otoczenia (sqft_living15)
    # grade w potędze lub jako ważny predyktor nieliniowy
    formula3 = "log_price ~ sqft_living + I(sqft_living**2) + grade + age + C(waterfront) + is_luxury + view + bathrooms + lat + sqft_living15 + sqft_living:grade"
    model3 = smf.ols(formula=formula3, data=houses)
    result3 = model3.fit()

## KROK 5: Ocena Wyników - Model Evaluation

In [ ]:
if not houses.empty:
    # print(result1.summary())
    # print(result2.summary())

    ## model 1
    forecast_model1 = result1.fittedvalues
    
    plt.figure(figsize=(8,6))
    plt.scatter(houses['price'], forecast_model1, alpha=0.5)
    plt.plot([houses['price'].min(), houses['price'].max()], [houses['price'].min(), houses['price'].max()], 'r--')
    plt.title("Model 1: Empirical vs Forecast")
    plt.xlabel("Empirical Price")
    plt.ylabel("Forecasted Price")
    plt.show()

    ## model 2
    # Model 2 prognozuje logarytm, więc musimy zrobić exp() aby porównać
    forecast_model2_log = result2.fittedvalues
    forecast_model2 = np.exp(forecast_model2_log)
    
    # Histogram reszt (Log Scale)
    plt.figure(figsize=(8,6))
    result2.resid.hist(bins=50) 
    plt.title("Model 2 Residuals Histogram (Log Scale)")
    plt.show()
    
    ## model 3
    forecast_model3_log = result3.fittedvalues
    forecast_model3 = np.exp(forecast_model3_log)

### Testy i Metryki

In [ ]:
if not houses.empty:
    # Próbkujemy, bo dla dużych danych Shapiro zawsze odrzuci H0
    sample_resid = result2.resid.sample(2000) if len(result2.resid) > 2000 else result2.resid
    stat, p = shapiro(sample_resid)
    print(f"\nShapiro-Wilk Test (Model 2 residuals): p-value = {p}")

    def mape(y_true, y_pred):
        return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

    rmse1 = eval_measures.rmse(houses['price'], forecast_model1)
    rmse2 = eval_measures.rmse(houses['price'], forecast_model2)
    rmse3 = eval_measures.rmse(houses['price'], forecast_model3)
    
    mape1 = mape(houses['price'], forecast_model1)
    mape2 = mape(houses['price'], forecast_model2)
    mape3 = mape(houses['price'], forecast_model3)

    print("\n" + "="*70)
    print(f"{'Metryka':<15} | {'Model 1':<15} | {'Model 2':<15} | {'Model 3 (New)':<15}")
    print("-" * 70)
    print(f"{'R-squared':<15} | {result1.rsquared:<15.4f} | {result2.rsquared:<15.4f} (log) | {result3.rsquared:<15.4f} (log)")
    print(f"{'RMSE':<15} | {rmse1:<15.2f} | {rmse2:<15.2f} | {rmse3:<15.2f}")
    print(f"{'MAPE (%)':<15} | {mape1:<15.2f} | {mape2:<15.2f} | {mape3:<15.2f}")
    print(f"{'AIC':<15} | {result1.aic:<15.2f} | {result2.aic:<15.2f} | {result3.aic:<15.2f}")
    print("="*70 + "\n")

## KROK 6: Predykcja dla nowych danych

In [ ]:
if not houses.empty:
    print("\n" + "="*90)
    print("PREDYKCJA CEN DLA RÓŻNYCH DOMÓW (MODEL 3)")
    print("-" * 90)

    # 1. Definiujemy dane dla kilku domów o różnych profilach
    # Średnie wartości z datasetu dla uzupełnienia brakujących (mniej istotnych) zmiennych
    mean_lat = houses['lat'].mean()
    mean_sqft15 = houses['sqft_living15'].mean()
    
    new_houses_data = {
        'opis': [
            'Mały domek, standardowy', 
            'Duży dom rodzinny, wysoki standard', 
            'Willa nad wodą, luksusowa', 
            'Stara rudera do remontu', 
            'Apartament w centrum (hipotetyczny)'
        ],
        'sqft_living': [1000, 2500, 4500, 1200, 1800],
        'grade': [7, 9, 11, 5, 8],          # 1-13 scale (7 is avg, 11 is excellent)
        'age': [10, 5, 2, 60, 15],
        'waterfront': [0, 0, 1, 0, 0],      # 0/1
        'is_luxury': [0, 0, 1, 0, 1],       # 0/1 (based on zipcode)
        'view': [0, 2, 4, 0, 1],            # 0-4 scale
        'bathrooms': [1, 2.5, 4.5, 1, 2],
        'lat': [mean_lat, mean_lat, mean_lat + 0.1, mean_lat - 0.05, mean_lat + 0.05], # Lokalizacja ma wpływ
        'sqft_living15': [1200, 2400, 4000, 1200, 2000] # Otoczenie
    }

    new_houses_df = pd.DataFrame(new_houses_data)

    # 2. Predykcja (Model 3 zwraca logarytm -> robimy exp)
    predicted_log = result3.predict(new_houses_df)
    new_houses_df['predicted_price'] = np.exp(predicted_log)

    # Wyświetlenie wyników
    print(f"{'Opis domu':<35} | {'Pow. (sqft)':<12} | {'Ocena':<6} | {'Woda':<5} | {'Przewidywana Cena ($)':<25}")
    print("-" * 90)

    for index, row in new_houses_df.iterrows():
        print(f"{row['opis']:<35} | {row['sqft_living']:<12} | {row['grade']:<6} | {row['waterfront']:<5} | {row['predicted_price']:<15,.2f}")

    print("="*90 + "\n")